# CEDS inventory and QA

Audits a CEDS release: file discovery, metadata integrity, time coverage,
source-id consistency, and truncated-download detection. Writes a provenance
manifest. **No aggregation happens here** — that is notebook 02.

Set `RELEASE` below to audit either release. The checks are identical for both,
which is what makes the old-vs-new comparison valid.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from slcf_forcings.ceds import (
    MAIN_SPECIES,
    assert_all_readable,
    audit_collection,
    check_source_consistency,
    check_time_coverage,
    flag_truncated,
    select_files,
    write_manifest,
)

# "archive"    = CEDS-CMIP-2025-03-18 (reference; known CO/CO2/NOx gridding issue)
# "production" = CEDS-CMIP-2025-04-18 (current CMIP7 DECK source id)
RELEASE = "archive"

DATA_DIRS = {
    "archive": PROJECT_ROOT / "data" / "SLCF_main" / "old",
    "production": PROJECT_ROOT / "data" / "SLCF_main",
}
DATA_DIR = DATA_DIRS[RELEASE]
OUTPUT_DIR = PROJECT_ROOT / "output" / RELEASE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Release:    {RELEASE}")
print(f"Data dir:   {DATA_DIR}")
print(f"Output dir: {OUTPUT_DIR}")

Release:    archive
Data dir:   /Users/assengu/Documents/projects/side05_cmip_feoc_forcings_slfc_evaluation/data/SLCF_main/old
Output dir: /Users/assengu/Documents/projects/side05_cmip_feoc_forcings_slfc_evaluation/output/archive


## 1. Metadata audit

Opens every file with `decode_times=False` — header reads only, no data.

In [2]:
audit = audit_collection(DATA_DIR)
audit.to_csv(OUTPUT_DIR / "audit.csv", index=False)

print(f"{len(audit)} emissions files discovered\n")
print(
    audit.groupby(["stream", "species"])
    .agg(n_files=("file", "count"), total_gb=("size_bytes", lambda s: s.sum() / 1e9))
    .round(2)
    .to_string()
)

104 emissions files discovered

                 n_files  total_gb
stream  species                   
AIR     BC             6      2.72
        CH4            2      0.02
        CO             6      2.73
        CO2            6      2.73
        N2O            2      1.46
        NH3            6      2.73
        NMVOC          6      2.73
        NOx            6      2.73
        OC             6      2.73
        SO2            6      1.39
surface BC             6      2.91
        CH4            2      0.73
        CO             6      2.94
        CO2            6      2.74
        N2O            2      1.12
        NH3            6      4.03
        NMVOC          6      4.22
        NOx            6      3.60
        OC             6      2.89
        SO2            6      2.94


## 2. Integrity checks

Each check is a question with a verifiable answer, not a table to eyeball.

In [3]:
unreadable = audit[~audit["readable"].fillna(False).astype(bool)]
print(f"Unreadable: {len(unreadable)} / {len(audit)}")
if not unreadable.empty:
    print(unreadable[["file", "size_bytes", "error"]].to_string(index=False))

suspect = flag_truncated(audit)
print(f"\nSuspiciously small for their chunk: {len(suspect)}")
if not suspect.empty:
    print(suspect[["file", "size_bytes"]].to_string(index=False))

Unreadable: 1 / 104
                                                                                file  size_bytes                                                                                                                                                                                                                 error
SO2-em-AIR-anthro_input4MIPs_emissions_CMIP_CEDS-CMIP-2025-03-18_gn_195001-199912.nc     5806108 [Errno -101] NetCDF: HDF error: '/Users/assengu/Documents/projects/side05_cmip_feoc_forcings_slfc_evaluation/data/SLCF_main/old/SO2-em-AIR-anthro_input4MIPs_emissions_CMIP_CEDS-CMIP-2025-03-18_gn_195001-199912.nc'

Suspiciously small for their chunk: 3
                                                                                file  size_bytes
CH4-em-AIR-anthro_input4MIPs_emissions_CMIP_CEDS-CMIP-2025-03-18_gn_197001-201912.nc    16991597
CH4-em-AIR-anthro_input4MIPs_emissions_CMIP_CEDS-CMIP-2025-03-18_gn_202001-202312.nc     1370796
    CO2-em-anthro_input4MIPs

In [4]:
mismatched = check_source_consistency(audit)
print(f"source_id mismatches: {len(mismatched)}")
if not mismatched.empty:
    print(mismatched[["file", "source_id_filename", "source_id"]].to_string(index=False))

short = check_time_coverage(audit)
print(f"\nTime-coverage mismatches: {len(short)}")
if not short.empty:
    print(short[["file", "time_size", "expected_months"]].to_string(index=False))

ok = audit[audit["readable"].fillna(False).astype(bool)]
print(f"\nUnits: {sorted(ok['units'].dropna().unique())}")
print(f"Sector counts (surface): "
      f"{sorted(ok.loc[ok['stream'] == 'surface', 'n_sectors'].unique())}")
print(f"Dimension signatures: {sorted(ok['dimensions'].unique())}")

source_id mismatches: 0

Time-coverage mismatches: 103
file  time_size  expected_months
 NaN        NaN              600
 NaN        NaN              600
 NaN        NaN              600
 NaN        NaN              600
 NaN        NaN              600
 NaN        NaN              288
 NaN        NaN              600
 NaN        NaN              600
 NaN        NaN              600
 NaN        NaN              600
 NaN        NaN              600
 NaN        NaN              288
 NaN        NaN              600
 NaN        NaN               48
 NaN        NaN              600
 NaN        NaN               48
 NaN        NaN              600
 NaN        NaN              600
 NaN        NaN              600
 NaN        NaN              600
 NaN        NaN              600
 NaN        NaN              288
 NaN        NaN              600
 NaN        NaN              600
 NaN        NaN              600
 NaN        NaN              600
 NaN        NaN              600
 NaN        NaN      

### Gate

Surface files are what notebook 02 aggregates, so they must all be readable.
The AIR stream is only needed for the pre-1970 extension (Phase 5), so it is
reported but does not block.

In [5]:
surface = audit[audit["stream"] == "surface"]
assert_all_readable(surface, context=f"surface stream, {RELEASE}")
print(f"PASS: all {len(surface)} surface files readable")

air_bad = audit[(audit["stream"] == "AIR") & (~audit["readable"].fillna(False))]
if not air_bad.empty:
    print(
        f"\nWARNING: {len(air_bad)} AIR file(s) unreadable. Not required for the "
        "1970-2023 baseline; re-fetch before Phase 5."
    )
    print(air_bad[["file", "size_bytes"]].to_string(index=False))

PASS: all 52 surface files readable

                                                                                file  size_bytes
SO2-em-AIR-anthro_input4MIPs_emissions_CMIP_CEDS-CMIP-2025-03-18_gn_195001-199912.nc     5806108


## 3. Species and time coverage

Selection is by parsed time coverage, not chunk-string matching. CEDS chunk
boundaries move between releases — March splits CH4/N2O at 197001-201912,
April at 197001-199912 — so a literal match returns different species sets for
the two releases.

In [6]:
WINDOWS = {
    "baseline_1970_2023": (1970, 2023),
    "working_2000_2023": (2000, 2023),
    "full_1750_2023": (1750, 2023),
}

for label, window in WINDOWS.items():
    try:
        files = select_files(DATA_DIR, MAIN_SPECIES, window)
        found = sorted({f.name.split("-em-")[0] for f in files})
        print(f"{label:22s} {len(files):3d} files, {len(found):2d} species")
    except ValueError as error:
        print(f"{label:22s} INCOMPLETE -- {error}")

baseline_1970_2023      20 files, 10 species
working_2000_2023       12 files, 10 species
full_1750_2023          52 files, 10 species


In [7]:
layout = (
    ok[ok["stream"] == "surface"]
    .sort_values(["species", "chunk_first_year"])
    [["species", "chunk_first_year", "chunk_last_year", "time_size", "n_sectors"]]
)
print(layout.to_string(index=False))

species  chunk_first_year  chunk_last_year  time_size  n_sectors
     BC              1750             1799      600.0        8.0
     BC              1800             1849      600.0        8.0
     BC              1850             1899      600.0        8.0
     BC              1900             1949      600.0        8.0
     BC              1950             1999      600.0        8.0
     BC              2000             2023      288.0        8.0
    CH4              1970             2019      600.0        8.0
    CH4              2020             2023       48.0        8.0
     CO              1750             1799      600.0        8.0
     CO              1800             1849      600.0        8.0
     CO              1850             1899      600.0        8.0
     CO              1900             1949      600.0        8.0
     CO              1950             1999      600.0        8.0
     CO              2000             2023      288.0        8.0
    CO2              1750

## 4. Provenance manifest

Closes the Phase 0 item: source id, version, coverage, licence and checksum per
input. Checksums are reused from the ESGF download manifest, which the download
wrapper already verified — that attests the bytes matched ESGF at download
time, a stronger claim than re-hashing local disk. Pass
`compute_checksums=True` to re-hash from disk instead.

In [8]:
download_manifest = DATA_DIR / "download_manifest_gn.tsv"
if not download_manifest.exists():
    download_manifest = DATA_DIR.parent / "download_manifest_gn.tsv"

manifest = write_manifest(
    DATA_DIR,
    OUTPUT_DIR / "data_manifest.csv",
    download_manifest=download_manifest if download_manifest.exists() else None,
    compute_checksums=False,
)

print(f"Wrote {len(manifest)} rows to {OUTPUT_DIR / 'data_manifest.csv'}\n")
print(manifest[["file", "source_id", "first_year", "last_year", "n_months",
                "sha256_source"]].head(10).to_string(index=False))

missing = manifest["sha256"].isna().sum()
if missing:
    print(f"\n{missing} file(s) without a checksum — no matching ESGF manifest entry.")


bad = manifest[~manifest["readable"]]
if not bad.empty:
    print(f"\n{len(bad)} file(s) recorded as unreadable:")
    print(bad[["file", "size_bytes", "error"]].to_string(index=False))

Manifest:   0%|          | 0/104 [00:00<?, ?file/s]

Wrote 104 rows to /Users/assengu/Documents/projects/side05_cmip_feoc_forcings_slfc_evaluation/output/archive/data_manifest.csv

                                                                               file            source_id  first_year  last_year  n_months sha256_source
BC-em-AIR-anthro_input4MIPs_emissions_CMIP_CEDS-CMIP-2025-03-18_gn_175001-179912.nc CEDS-CMIP-2025-03-18        1750       1799     600.0 esgf_manifest
BC-em-AIR-anthro_input4MIPs_emissions_CMIP_CEDS-CMIP-2025-03-18_gn_180001-184912.nc CEDS-CMIP-2025-03-18        1800       1849     600.0 esgf_manifest
BC-em-AIR-anthro_input4MIPs_emissions_CMIP_CEDS-CMIP-2025-03-18_gn_185001-189912.nc CEDS-CMIP-2025-03-18        1850       1899     600.0 esgf_manifest
BC-em-AIR-anthro_input4MIPs_emissions_CMIP_CEDS-CMIP-2025-03-18_gn_190001-194912.nc CEDS-CMIP-2025-03-18        1900       1949     600.0 esgf_manifest
BC-em-AIR-anthro_input4MIPs_emissions_CMIP_CEDS-CMIP-2025-03-18_gn_195001-199912.nc CEDS-CMIP-2025-03-18        